In [1]:
import sys

import torch

import pandas as pd

from config.feature_config import FeatureConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from dice4el.scenario.scenario_handler import ScenarioHandler

from dice4el.scenario.scenario_model import ScenarioLSTM
from dice4el.scenario.scenario_model_wrapper import ScenarioModelWrapper

from dice4el.dice4el_config import EventLogDiCEConfig
from dice4el.eventlog_dice import EventLogDiCE
from dice4el.eventlog_dice_optimized import EventLogDiCEOptimized

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=777)

In [3]:
df = pd.read_excel(
    "../../../../data/bpic20_Int.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Permit OrganizationalEntity": "string",
        "case:Amount": "float32",
        "case:RequestedAmount": "float32",
        "case:OriginalAmount": "float32",
        "case:Permit RequestedBudget": "float32",
        "case:AdjustedAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,case:AdjustedAmount,case:Amount,case:OriginalAmount,case:Permit OrganizationalEntity,case:Permit RequestedBudget,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
0,declaration 1002,2018-03-01 10:55:17,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Permit SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,declaration 1002,2018-03-01 10:55:21,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Permit APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,4.0
2,declaration 1002,2018-03-01 15:01:48,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Permit FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,14787.0
3,declaration 1002,2018-03-19 00:00:00,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Start trip,STAFF MEMBER,EMPLOYEE,1501092.0
4,declaration 1002,2018-03-23 00:00:00,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,End trip,STAFF MEMBER,EMPLOYEE,345600.0
5,declaration 1002,2018-03-27 16:15:02,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,404102.0
6,declaration 1002,2018-04-03 17:07:56,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Declaration APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,607974.0
7,declaration 1002,2018-04-05 09:45:53,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Declaration FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,146277.0
8,declaration 1002,2018-04-05 17:25:23,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Request Payment,SYSTEM,UNDEFINED,27570.0
9,declaration 1002,2018-04-09 17:30:58,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Payment Handled,SYSTEM,UNDEFINED,345935.0


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:AdjustedAmount', 'case:Amount', 'case:OriginalAmount', 'case:Permit OrganizationalEntity', 'case:Permit RequestedBudget', 'case:RequestedAmount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [19.00, 518400.00]                       112149.0000 quantile_derived    
case:Amount                    continuous     case     yes    [28.52, 1883.08]                         375.8531   quantile_derived    
case:RequestedAmount           continuous     case    

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

In [11]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [12]:
scenario_model = ScenarioLSTM.load(
    path = "../pretrained_models/"
)

In [13]:
scenario_model_wrapper = ScenarioModelWrapper(
    scenario_model=scenario_model,
    scenario_handler=scenario_handler,
    device=device
)

### --- Process Constraints ---

In [14]:
engine = ProcessModelConstraintEngine.load(
     path = "../../pretrained_models/"
)

In [15]:
engine.parallel_sets

[{'Permit APPROVED by SUPERVISOR', 'Permit FINAL_APPROVED by DIRECTOR'},
 {'Permit APPROVED by PRE_APPROVER', 'Permit FINAL_APPROVED by SUPERVISOR'}]

In [16]:
engine.branching_sets

[{'Declaration APPROVED by ADMINISTRATION',
  'Declaration APPROVED by BUDGET OWNER',
  'Declaration APPROVED by PRE_APPROVER',
  'Declaration APPROVED by SUPERVISOR',
  'Declaration FINAL_APPROVED by DIRECTOR',
  'Declaration FINAL_APPROVED by SUPERVISOR',
  'Declaration REJECTED by ADMINISTRATION',
  'Declaration REJECTED by BUDGET OWNER',
  'Declaration REJECTED by DIRECTOR',
  'Declaration REJECTED by EMPLOYEE',
  'Declaration REJECTED by MISSING',
  'Declaration REJECTED by PRE_APPROVER',
  'Declaration REJECTED by SUPERVISOR',
  'Declaration SUBMITTED by EMPLOYEE',
  'End trip',
  'Payment Handled',
  'Permit APPROVED by ADMINISTRATION',
  'Permit APPROVED by BUDGET OWNER',
  'Permit APPROVED by PRE_APPROVER',
  'Permit APPROVED by SUPERVISOR',
  'Permit FINAL_APPROVED by DIRECTOR',
  'Permit FINAL_APPROVED by SUPERVISOR',
  'Permit REJECTED by ADMINISTRATION',
  'Permit REJECTED by BUDGET OWNER',
  'Permit REJECTED by DIRECTOR',
  'Permit REJECTED by EMPLOYEE',
  'Permit REJECTE

### --- Load Experiments ---

In [17]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Int-cf_seed777_experiments_dice4el_output.txt", console=False)

In [18]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [19]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [20]:
dice4el_config = EventLogDiCEConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
    w_margin_loss=1.0,
    w_scenario_loss=1.0,
    w_distance_loss=1.0,
    w_cat_loss=1.0,
)
dice4el_config.validate()

In [21]:
cf_DiCE4EL = EventLogDiCE(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results = generator.run_experiment_df(
    cf_method=cf_DiCE4EL,
    technique="DiCE4EL_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/400 [00:00<?, ?case/s]

In [22]:
results

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,declaration 69274,6,1,0,0.127521,0.055042,0.2,0.666667,0.400000,...,1.194187,0.400000,0.127521,0.2,0.055042,0.666667,0.000000,0.000000,0.000000,0.000000
1,0,declaration 916,7,1,0,0.073854,0.147708,0.0,0.583333,0.411765,...,1.841810,0.411765,0.073854,0.0,0.147708,0.583333,0.772858,0.772858,0.999999,0.999999
2,0,declaration 54601,8,1,0,0.035699,0.071398,0.0,0.583333,0.157895,...,1.537266,0.157895,0.035699,0.0,0.071398,0.583333,0.760339,0.760339,0.999999,0.999999
3,0,declaration 30359,9,1,0,0.038328,0.076655,0.0,0.583333,0.476190,...,1.186247,0.476190,0.038328,0.0,0.076655,0.583333,0.088395,0.088395,0.000000,0.000000
4,0,declaration 34740,10,1,0,0.062625,0.125249,0.0,0.583333,0.130435,...,1.403798,0.130435,0.062625,0.0,0.125249,0.583333,0.627405,0.627405,0.999998,0.999998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,38,declaration 56837,21,1,0,0.008007,0.016014,0.0,0.416667,0.777778,...,2.152534,0.777778,0.008007,0.0,0.016014,0.416667,0.950082,0.000000,1.000000,0.000000
348,38,declaration 51709,21,1,0,0.073242,0.146484,0.0,0.416667,0.377778,...,1.427936,0.377778,0.073242,0.0,0.146484,0.416667,0.560249,0.560249,0.999995,0.999995
349,38,declaration 20507,22,1,0,0.087754,0.175508,0.0,0.416667,0.787234,...,2.231376,0.787234,0.087754,0.0,0.175508,0.416667,0.939721,0.893270,1.000000,1.000000
350,38,declaration 4604,23,1,0,0.183098,0.366197,0.0,0.416667,0.469388,...,1.768649,0.469388,0.183098,0.0,0.366197,0.416667,0.699496,0.699496,0.999999,0.999999


In [23]:
cf_DiCE4EL_optim = EventLogDiCEOptimized(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results_optim = generator.run_experiment_df(
    cf_method=cf_DiCE4EL_optim,
    technique="DiCE4EL-Optimized_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/400 [00:00<?, ?case/s]

In [24]:
results_optim

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,declaration 69274,6,1,0,0.158399,0.116797,0.200000,0.666667,0.400000,...,1.225065,0.400000,0.158399,0.200000,0.116797,0.666667,0.000000,0.000000,0.0,0.0
1,0,declaration 916,7,1,0,0.222640,0.045279,0.400000,0.750000,0.470588,...,1.778611,0.470588,0.222640,0.400000,0.045279,0.750000,0.335383,0.335383,0.0,0.0
2,0,declaration 54601,8,1,0,0.131141,0.062282,0.200000,0.666667,0.526316,...,1.782420,0.526316,0.131141,0.200000,0.062282,0.666667,0.458297,0.458297,0.0,0.0
3,0,declaration 30359,9,1,0,0.121241,0.042481,0.200000,0.666667,0.476190,...,1.733362,0.476190,0.121241,0.200000,0.042481,0.666667,0.469265,0.469265,0.0,0.0
4,0,declaration 34740,10,1,0,0.245020,0.090040,0.400000,0.750000,0.608696,...,1.912183,0.608696,0.245020,0.400000,0.090040,0.750000,0.308467,0.308467,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,38,declaration 56837,21,1,0,0.016262,0.032524,0.000000,0.416667,0.777778,...,2.160838,0.777778,0.016262,0.000000,0.032524,0.416667,0.950132,0.000000,1.0,0.0
348,38,declaration 51709,21,1,0,0.229331,0.268186,0.190476,0.527778,0.822222,...,2.513966,0.822222,0.229331,0.190476,0.268186,0.527778,0.934635,0.432274,1.0,0.0
349,38,declaration 20507,22,1,0,0.167035,0.286450,0.047619,0.444444,0.574468,...,2.143081,0.574468,0.167035,0.047619,0.286450,0.444444,0.957134,0.000000,1.0,0.0
350,38,declaration 4604,23,1,0,0.252694,0.362531,0.142857,0.500000,0.918367,...,2.623970,0.918367,0.252694,0.142857,0.362531,0.500000,0.952909,0.000000,1.0,0.0


### --- Cleanup ---

In [25]:
sys.stdout = original_stdout
log_file.close()